# 실습 1 — 벡터 검색 창고 만들기

PDF 로딩 → 청킹 → BGE-M3 임베딩 → FAISS → Qdrant → 검색 비교 + 메타데이터 필터

**참고자료 코드 대비 수정 사항**
- `langchain.text_splitter` → `langchain_text_splitters` (langchain 1.x에서 모듈 제거)
- `client.search()` → `client.query_points()` (qdrant-client 1.16+에서 제거)
- PDF 16개를 한 번에 로딩하고, payload에 `source`(파일명)·`doc_type`(발행처)을 추가해 문서 단위 필터 가능
- 청크/임베딩을 `cache/`에 저장해 실습 2에서 재사용

**사전 준비**: Qdrant 컨테이너 실행 중이어야 합니다 → http://localhost:6333/dashboard

## Step 0. 설정

In [ ]:
import os
# macOS: torch와 faiss가 각자 libomp를 불러와 커널이 죽는 문제 방지 (torch/faiss import 전에 설정)
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'

from pathlib import Path
import json, time
import numpy as np

BASE_DIR = Path.cwd()
PDF_DIR = BASE_DIR / 'PDF'
CACHE_DIR = BASE_DIR / 'cache'
CACHE_DIR.mkdir(exist_ok=True)

# True: PDF 16개 전체 (청크 약 4~5천 개, 임베딩에 시간이 꽤 걸림)
# False: 반도체 보고서 3종 + SPRi 최신호만 (빠르게 흐름 확인용)
USE_ALL_DOCS = False

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
COLLECTION = 'rag_docs'

# 파일명 앞 번호 → 발행처 태그
DOC_TYPES = {'1': 'SPRi', '2': 'ETRI', '3': '삼일PwC', '4': '반도체특위', '5': 'AI반도체동향'}

pdf_files = sorted(PDF_DIR.glob('*.pdf'))
if not USE_ALL_DOCS:
    pdf_files = [p for p in pdf_files
                 if p.name[0] in ('3', '4', '5') or '202606' in p.name]
for p in pdf_files:
    print(p.name)
print(f'\n대상 문서: {len(pdf_files)}개')

## Step 1. PDF 로딩 (PyMuPDF)

In [ ]:
import fitz  # PyMuPDF

def load_pdf(path):
    doc = fitz.open(path)
    pages = []
    for i, page in enumerate(doc):
        text = page.get_text()
        if text.strip():
            pages.append({
                'text': text,
                'page': i + 1,                       # int로 저장해야 Range 필터가 동작
                'source': path.name,
                'doc_type': DOC_TYPES.get(path.name[0], '기타'),
            })
    return pages

pages = []
for path in pdf_files:
    p = load_pdf(path)
    print(f'{len(p):4d}페이지 | {path.name}')
    pages.extend(p)
print(f'\n총 {len(pages)}페이지 로딩 완료')
print(pages[0]['text'][:300])

## Step 2. 청킹 (RecursiveCharacterTextSplitter)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', '.', ' '])

chunks = []
for page in pages:
    for split in splitter.split_text(page['text']):
        chunks.append({'text': split, 'page': page['page'],
                       'source': page['source'], 'doc_type': page['doc_type']})

lengths = [len(c['text']) for c in chunks]
print(f'총 청크 수: {len(chunks)}')
print(f'청크 길이 평균 {np.mean(lengths):.0f}자 / 최대 {max(lengths)}자')

## Step 3. 임베딩 생성 (BGE-M3)

최초 실행 시 모델(약 2.3GB)을 다운로드합니다. Apple Silicon에서는 MPS(GPU)를 사용합니다.
같은 문서·청크 설정으로 이미 만든 임베딩이 `cache/`에 있으면 재사용합니다.

In [ ]:
import torch
from FlagEmbedding import BGEM3FlagModel

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=(device != 'cpu'), devices=device)
print('device:', device)

In [ ]:
tag = f"{'all' if USE_ALL_DOCS else 'sample'}_{CHUNK_SIZE}_{CHUNK_OVERLAP}"
emb_path = CACHE_DIR / f'embeddings_{tag}.npy'
chunk_path = CACHE_DIR / f'chunks_{tag}.json'

if emb_path.exists() and chunk_path.exists() and \
        json.loads(chunk_path.read_text()) == chunks:
    embeddings = np.load(emb_path)
    print('캐시에서 임베딩 로드')
else:
    texts = [c['text'] for c in chunks]
    t0 = time.time()
    output = model.encode(texts, batch_size=16, max_length=512, return_dense=True)
    embeddings = np.array(output['dense_vecs']).astype('float32')
    print(f'임베딩 소요: {time.time() - t0:.1f}초')
    np.save(emb_path, embeddings)
    chunk_path.write_text(json.dumps(chunks, ensure_ascii=False))

print(f'shape: {embeddings.shape}')  # (N, 1024)

## Step 4. FAISS 인덱스 구축 및 검색

In [ ]:
import faiss

def encode_query(query):
    return model.encode([query])['dense_vecs'].astype('float32')

index = faiss.IndexHNSWFlat(1024, 32)
index.hnsw.efConstruction = 200
index.add(embeddings)
print('FAISS 벡터 수:', index.ntotal)

query = 'HBM이란 무엇인가?'
q_vec = encode_query(query)
dists, idxs = index.search(q_vec, k=5)
for i, (d, idx) in enumerate(zip(dists[0], idxs[0])):
    c = chunks[idx]
    print(f"{i+1}위 (L2={d:.3f}) [{c['source']} p.{c['page']}] {c['text'][:80]!r}")

## Step 5~6. Qdrant Collection 생성 + Upsert

FAISS와 달리 벡터 옆에 `text`, `page`, `source`, `doc_type`을 payload로 같이 저장합니다.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams, Distance, PointStruct, PayloadSchemaType)

client = QdrantClient('localhost', port=6333)

# 재실행해도 에러 나지 않도록 있으면 지우고 다시 생성
if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)
client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE))

# 필터에 쓸 필드는 payload 인덱스를 만들어두면 빨라집니다
client.create_payload_index(COLLECTION, 'page', PayloadSchemaType.INTEGER)
client.create_payload_index(COLLECTION, 'doc_type', PayloadSchemaType.KEYWORD)
client.create_payload_index(COLLECTION, 'source', PayloadSchemaType.KEYWORD)

points = [PointStruct(id=i, vector=embeddings[i].tolist(), payload=chunks[i])
          for i in range(len(chunks))]
BATCH = 256
for s in range(0, len(points), BATCH):
    client.upsert(COLLECTION, points[s:s + BATCH])

print('Qdrant 포인트 수:', client.count(COLLECTION).count)
print('대시보드: http://localhost:6333/dashboard')

## Step 7. 검색 비교 — FAISS vs Qdrant

In [ ]:
def qdrant_search(query, limit=5, query_filter=None):
    return client.query_points(
        collection_name=COLLECTION,
        query=encode_query(query)[0].tolist(),
        query_filter=query_filter,
        limit=limit,
        with_payload=True,
    ).points

def show(points):
    for i, p in enumerate(points):
        pl = p.payload
        print(f"{i+1}위 (cos={p.score:.3f}) [{pl['source']} p.{pl['page']}] {pl['text'][:80]!r}")

def compare(query, k=5):
    print(f'■ 질문: {query}')
    _, f_ids = index.search(encode_query(query), k)
    q_points = qdrant_search(query, k)
    print('-- FAISS')
    for i, idx in enumerate(f_ids[0]):
        c = chunks[idx]
        print(f"{i+1}위 [{c['source']} p.{c['page']}] {c['text'][:80]!r}")
    print('-- Qdrant')
    show(q_points)
    overlap = set(map(int, f_ids[0])) & {p.id for p in q_points}
    print(f'겹치는 결과: {len(overlap)}/{k}\n')

compare('HBM이란 무엇인가?')

## Step 8. 메타데이터 필터 검색

In [ ]:
from qdrant_client.models import Filter, FieldCondition, Range, MatchValue

query = 'HBM이란 무엇인가?'

print('■ 필터 없음'); show(qdrant_search(query))

# 슬라이드 예시는 gte=5 — 상위 결과가 이미 모두 5페이지 이후라 차이가 안 보여서 20으로 설정
print('\n■ 20페이지 이후만')
show(qdrant_search(query, query_filter=Filter(must=[
    FieldCondition(key='page', range=Range(gte=20))])))

print('\n■ 반도체특위 보고서에서만')
show(qdrant_search(query, query_filter=Filter(must=[
    FieldCondition(key='doc_type', match=MatchValue(value='반도체특위'))])))

## 실습 질문 (Easy → Hard)

In [ ]:
questions = [
    'HBM이란 무엇인가?',
    'AI 반도체 시장을 견인하는 요인은?',
    '한국의 반도체 경쟁력은?',
    '보고서들을 종합해 생성형 AI가 요구하는 핵심 반도체 기술과, 한국이 우선 강화해야 할 분야를 설명하시오',
]
for q in questions:
    print(f'■ {q}')
    points = qdrant_search(q)
    show(points)
    print('참조 문서:', sorted({p.payload['doc_type'] for p in points}), '\n')

## 보너스 — 청크 크기 바꿔서 비교

Step 0의 `CHUNK_SIZE`/`CHUNK_OVERLAP`을 바꾸고 Step 2부터 다시 실행해 보세요 (예: 200/20, 1000/100).
캐시 파일명에 설정값이 들어가므로 이전 결과는 그대로 남습니다.
대시보드(http://localhost:6333/dashboard)에서 컬렉션의 포인트와 payload도 직접 살펴보세요.